# Operation Chaos Commute — Data Exploration

## Members: Shirley (Shuzhou) Li, Wenjun Yao, Sikandar Ali

In this project, we analyze Citi Bike trip data to understand commuting patterns in an urban environment. We begin with basic data exploration to understand the structure, quality, and insihgts of the dataset.

---

The data range for this exploration will be increase to 202102 ~ 202602, about 5 years worth of data. Link to the datasource: https://citibikenyc.com/system-data

### Part 1: Data Loading

In [1]:
import pandas as pd
import numpy as np

df_list = []

for year in range(2021, 2027):
    for month in range(1, 13):
        month_str = str(month).zfill(2)
        file = f"data/JC-{year}{month_str}-citibike-tripdata.csv"

        try:
            temp = pd.read_csv(file)
            df_list.append(temp)
        except:
            # skip files that don't exist
            # however from manual inspection, all files from 202102~202602 exist
            pass

df = pd.concat(df_list, ignore_index=True)

df.head(10)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,121DD7DD23CB1335,docked_bike,2021-02-03 23:11:28,2021-02-03 23:18:28,Hoboken Ave at Monmouth St,JC105,Christ Hospital,JC034,40.735208,-74.046964,40.734786,-74.050444,member
1,FD73FB85F008349D,docked_bike,2021-02-27 16:34:05,2021-02-27 16:56:40,Newport Pkwy,JC008,Marin Light Rail,JC013,40.728744,-74.032108,40.714584,-74.042817,member
2,39F9E6663CB5FDF6,docked_bike,2021-02-26 23:16:04,2021-02-26 23:22:25,Journal Square,JC103,Baldwin at Montgomery,JC020,40.733670,-74.062500,40.723659,-74.064194,member
3,A64745CB0792EC6F,docked_bike,2021-02-24 16:51:50,2021-02-24 17:16:09,Hoboken Ave at Monmouth St,JC105,Hoboken Ave at Monmouth St,JC105,40.735208,-74.046963,40.735208,-74.046964,casual
4,75CC76EB9543764A,docked_bike,2021-02-24 20:44:16,2021-02-24 20:44:46,Hoboken Ave at Monmouth St,JC105,Hoboken Ave at Monmouth St,JC105,40.735208,-74.046963,40.735208,-74.046964,member
5,9E512D211620A136,docked_bike,2021-02-12 20:03:24,2021-02-12 20:18:34,Newport Pkwy,JC008,Paulus Hook,JC002,40.728744,-74.032108,40.714145,-74.033552,casual
6,381EF1361C1A6DB8,docked_bike,2021-02-16 19:39:39,2021-02-16 19:41:59,Newark Ave,JC032,Monmouth and 6th,JC075,40.721525,-74.046305,40.725685,-74.048790,member
7,928FA29F84E99326,docked_bike,2021-02-12 18:14:07,2021-02-12 18:19:49,Newport Pkwy,JC008,Warren St,JC006,40.728744,-74.032108,40.721124,-74.038051,member
8,A7E2FCAC8E229A75,docked_bike,2021-02-16 05:17:11,2021-02-16 05:22:09,Newport Pkwy,JC008,Warren St,JC006,40.728744,-74.032108,40.721124,-74.038051,member
9,A591312D7A1533DD,docked_bike,2021-02-10 11:44:47,2021-02-10 11:52:30,Newport Pkwy,JC008,Warren St,JC006,40.728744,-74.032108,40.721124,-74.038051,member


**Note:** "lat" is latitude and "lng" is longtitude.

### Part 2: Basic Info

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4529661 entries, 0 to 4529660
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             str    
 1   rideable_type       str    
 2   started_at          str    
 3   ended_at            str    
 4   start_station_name  str    
 5   start_station_id    str    
 6   end_station_name    str    
 7   end_station_id      str    
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       str    
dtypes: float64(4), str(9)
memory usage: 449.3 MB


In [3]:
df.describe()

,start_lat,start_lng,end_lat,end_lng
count,4.529658e+06,4.529658e+06,4.519794e+06,4.519794e+06
mean,4.073199e+01,-7.404041e+01,4.073194e+01,-7.404014e+01
std,1.219500e-02,1.214665e-02,1.230413e-02,1.221195e-02
min,4.067833e+01,-7.410810e+01,4.062000e+01,-7.423000e+01
25%,4.072112e+01,-7.404595e+01,4.072112e+01,-7.404557e+01
50%,4.073479e+01,-7.403798e+01,4.073479e+01,-7.403798e+01
75%,4.074226e+01,-7.403109e+01,4.074226e+01,-7.403097e+01
max,4.086394e+01,-7.394117e+01,4.092000e+01,-7.384000e+01


**Interpretation**: Since all data is located in NYC, the std is quite small. The minimum and maximum of latittude and longtitue are also quite similar which aligns the geographical region of the data (only NYC).

In [4]:
df.isnull().sum().sort_values(ascending=False)

end_station_id        18611
end_station_name      16741
end_lat                9867
end_lng                9867
start_station_name      203
start_station_id        203
start_lat                 3
start_lng                 3
ride_id                   0
rideable_type             0
started_at                0
ended_at                  0
member_casual             0
dtype: int64

### Part 3: Dealing with Null/Missing Values

In [5]:
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
}).sort_values(by="missing_percent", ascending=False)

missing_summary

,missing_count,missing_percent
end_station_id,18611,0.410870
end_station_name,16741,0.369586
end_lat,9867,0.217831
end_lng,9867,0.217831
start_station_name,203,0.004482
start_station_id,203,0.004482
start_lat,3,0.000066
start_lng,3,0.000066
ride_id,0,0.000000
rideable_type,0,0.000000


**Interpretation:** Because the percentage of missing values are quite low, below 0.5%, so for simplicity, I will drop all missing values.

In [6]:
df = df.dropna()

In [7]:
df.isnull().sum()

ride_id               0
rideable_type         0
started_at            0
ended_at              0
start_station_name    0
start_station_id      0
end_station_name      0
end_station_id        0
start_lat             0
start_lng             0
end_lat               0
end_lng               0
member_casual         0
dtype: int64

### Part 4: Converting Columns to DateTime Objects & Supplemental Variable Creations

I am converting "started_at" anf "ended_at" to datetime objects, adding new columns called "tripduration", "year", "month", "day_of_week", "hour", "is_weekend", "distance", and "speed".

In [8]:
#convert time columns to datetime
df["started_at"] = pd.to_datetime(df["started_at"], format="mixed", errors="coerce")
df["ended_at"] = pd.to_datetime(df["ended_at"], format="mixed", errors="coerce")

# calculate trip duration variable in minutes
df["tripduration"] = (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60

# add year and month variable by extraction
df["year"] = df["started_at"].dt.year
df["month"] = df["started_at"].dt.month

df[["started_at", "ended_at", "tripduration", "year", "month"]].head()

,started_at,ended_at,tripduration,year,month
0,2021-02-03 23:11:28,2021-02-03 23:18:28,7.000000,2021,2
1,2021-02-27 16:34:05,2021-02-27 16:56:40,22.583333,2021,2
2,2021-02-26 23:16:04,2021-02-26 23:22:25,6.350000,2021,2
3,2021-02-24 16:51:50,2021-02-24 17:16:09,24.316667,2021,2
4,2021-02-24 20:44:16,2021-02-24 20:44:46,0.500000,2021,2


In [9]:
#Adding more supplemental variables
df["day_of_week"] = df["started_at"].dt.day_name()
df["hour"] = df["started_at"].dt.hour
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"])

df["distance"] = np.sqrt(
    (df["end_lat"] - df["start_lat"])**2 +
    (df["end_lng"] - df["start_lng"])**2
)

df = df[df["tripduration"] > 0]
df["speed"] = (df["distance"]/df["tripduration"])

df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,end_lng,member_casual,tripduration,year,month,day_of_week,hour,is_weekend,distance,speed
0,121DD7DD23CB1335,docked_bike,2021-02-03 23:11:28,2021-02-03 23:18:28,Hoboken Ave at Monmouth St,JC105,Christ Hospital,JC034,40.735208,-74.046964,...,-74.050444,member,7.000000,2021,2,Wednesday,23,False,3.505493e-03,5.007848e-04
1,FD73FB85F008349D,docked_bike,2021-02-27 16:34:05,2021-02-27 16:56:40,Newport Pkwy,JC008,Marin Light Rail,JC013,40.728744,-74.032108,...,-74.042817,member,22.583333,2021,2,Saturday,16,True,1.775355e-02,7.861350e-04
2,39F9E6663CB5FDF6,docked_bike,2021-02-26 23:16:04,2021-02-26 23:22:25,Journal Square,JC103,Baldwin at Montgomery,JC020,40.733670,-74.062500,...,-74.064194,member,6.350000,2021,2,Friday,23,False,1.015346e-02,1.598970e-03
3,A64745CB0792EC6F,docked_bike,2021-02-24 16:51:50,2021-02-24 17:16:09,Hoboken Ave at Monmouth St,JC105,Hoboken Ave at Monmouth St,JC105,40.735208,-74.046963,...,-74.046964,casual,24.316667,2021,2,Wednesday,16,False,8.421525e-07,3.463273e-08
4,75CC76EB9543764A,docked_bike,2021-02-24 20:44:16,2021-02-24 20:44:46,Hoboken Ave at Monmouth St,JC105,Hoboken Ave at Monmouth St,JC105,40.735208,-74.046963,...,-74.046964,member,0.500000,2021,2,Wednesday,20,False,8.421525e-07,1.684305e-06


### Part 5: Variable & Pattern Inspections

**Start & End Station Popularity**

In [10]:
for col in ["start_station_name", "end_station_name"]:
    counts = df[col].value_counts()

    df_out = pd.DataFrame({
        "count": counts,
        "percent": (counts / len(df) * 100).round(2)
    })

    print(f"\nTop {col.replace('_', ' ').title()}:")
    display(df_out.head(10))


Top Start Station Name:


,count,percent
start_station_name,,
Grove St PATH,208075,4.61
Hoboken Terminal - River St & Hudson Pl,163372,3.62
Hoboken Terminal - Hudson St & Hudson Pl,130246,2.89
South Waterfront Walkway - Sinatra Dr & 1 St,128434,2.85
Newport PATH,104811,2.32
Hamilton Park,103242,2.29
Newport Pkwy,100099,2.22
City Hall - Washington St & 1 St,94319,2.09
11 St & Washington St,88218,1.96



Top End Station Name:


,count,percent
end_station_name,,
Grove St PATH,224435,4.98
Hoboken Terminal - River St & Hudson Pl,163273,3.62
Hoboken Terminal - Hudson St & Hudson Pl,132424,2.94
South Waterfront Walkway - Sinatra Dr & 1 St,129081,2.86
Newport PATH,105075,2.33
Hamilton Park,104020,2.31
Newport Pkwy,100185,2.22
City Hall - Washington St & 1 St,95418,2.12
11 St & Washington St,88012,1.95


**Round Trip Detection**

This is a work in progress, this current method does not account for ride ID, date, and other variables.

In [11]:
round_trips = (df["start_station_name"] == df["end_station_name"]).sum()
total_trips = len(df)

round_trip_pct = round_trips / total_trips * 100

round_trips, round_trip_pct

(np.int64(279834), np.float64(6.203716102074424))

**User Type Distribution**

In [12]:
user_counts = df["member_casual"].value_counts()
user_pct = df["member_casual"].value_counts(normalize=True) * 100

user_counts
user_pct

member_casual
member    72.166213
casual    27.833787
Name: proportion, dtype: float64

**Most Busy Hours**

In [13]:
hour_counts = df["hour"].value_counts().sort_index()
hour_pct = (hour_counts / len(df) * 100)

df_out = pd.DataFrame({
        "count": hour_counts,
        "percent": hour_pct
    })

df_out

,count,percent
hour,,
0,54094,1.199225
1,33556,0.743912
2,20894,0.463205
3,11764,0.260799
4,14709,0.326088
5,40461,0.896991
6,116101,2.573875
7,223676,4.958734
8,292244,6.478837


**Interpretation:** This confirms the convention of Rush Hour. 17 and 18 are the busiest. 8, 15, 16, and 19 are also quite busy. 

**Rideable Type distribution**

In [14]:
ride_counts = df["rideable_type"].value_counts()
ride_pct = df["rideable_type"].value_counts(normalize=True) * 100

ride_counts, ride_pct

(rideable_type
 classic_bike     2575400
 electric_bike    1794908
 docked_bike       140440
 Name: count, dtype: int64,
 rideable_type
 classic_bike     57.094743
 electric_bike    39.791804
 docked_bike       3.113453
 Name: proportion, dtype: float64)

**Busiest Month, Day of the Week, Weekend vs Weekday**

In [15]:
month_counts = df["month"].value_counts().sort_index()
month_counts

month
1     222409
2     213665
3     269916
4     326726
5     411016
6     478237
7     407536
8     533852
9     525302
10    496373
11    368699
12    257017
Name: count, dtype: int64

**Interpretation:** August is the busiest month. The distribution also presents a strong seasonal influerence on ridership.

In [16]:
day_counts = df["day_of_week"].value_counts()
day_counts

day_of_week
Friday       682789
Thursday     677333
Wednesday    676898
Tuesday      660709
Saturday     638530
Monday       607955
Sunday       566534
Name: count, dtype: int64

**Interpretation:** Needs further investigation, but this busiest day of the week is potentially caused by the social calender.

In [17]:
weekend = df["day_of_week"].isin(["Saturday", "Sunday"])

weekend_counts = weekend.value_counts()
weekend_pct = weekend.value_counts(normalize=True) * 100

weekend_counts, weekend_pct

(day_of_week
 False    3305684
 True     1205064
 Name: count, dtype: int64,
 day_of_week
 False    73.284608
 True     26.715392
 Name: proportion, dtype: float64)

**Interpretation:** The percentages of rides among weekdays and weekends are generally proportional. 100%/7*5 = 71.43% vs 73.28% is not a huge difference.